In [ ]:
from __future__ import annotations
import gc, json, os, random, shutil
from dataclasses import asdict, dataclass
from typing import Dict, List

import joblib, numpy as np, pandas as pd, tensorflow as tf, tensorflow_io as tfio
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from xgboost.spark import SparkXGBRegressor

def stage(name: str): print(f"\n{'=' * 100}\n{name}\n{'=' * 100}")
def mkdirs(*paths: str): [os.makedirs(p, exist_ok=True) for p in paths]
def reset_dir(path: str): shutil.rmtree(path, ignore_errors=True); os.makedirs(path, exist_ok=True)
def seed_all(seed: int): random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
def nonneg(x: np.ndarray) -> np.ndarray: return np.maximum(np.asarray(x, dtype=np.float32), 0.0)

def cleanup(spark: SparkSession | None = None, *objs: object, clear_tf: bool = False):
    for o in objs:
        try: o.unpersist(blocking=True)
        except Exception: pass
    if spark is not None:
        for fn in (spark.catalog.clearCache, lambda: spark._jvm.java.lang.System.gc()):
            try: fn()
            except Exception: pass
    if clear_tf:
        try: tf.keras.backend.clear_session()
        except Exception: pass
    gc.collect()

def metrics(df: DataFrame, pred: str, y: str) -> Dict[str, float | None]:
    row = df.select(
        F.avg(F.abs(F.col(y) - F.col(pred))).alias("MAE"),
        F.sqrt(F.avg(F.pow(F.col(y) - F.col(pred), 2))).alias("RMSE"),
        (F.avg(F.when(F.col(y) > 0, F.abs((F.col(y) - F.col(pred)) / F.col(y)))) * 100).alias("MAPE_nonzero"),
        (F.avg(2 * F.abs(F.col(pred) - F.col(y)) / (F.abs(F.col(y)) + F.abs(F.col(pred)) + F.lit(1e-6))) * 100).alias("sMAPE"),
    ).first()
    return {k: float(row[k]) if row[k] is not None else None for k in ("MAE", "RMSE", "MAPE_nonzero", "sMAPE")}

def copy_dir_to_hdfs(spark: SparkSession, local_dir: str, hdfs_dir: str):
    jvm = spark._jvm
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    dst = jvm.org.apache.hadoop.fs.Path(hdfs_dir)
    if fs.exists(dst): fs.delete(dst, True)
    fs.mkdirs(dst)
    for name in sorted(os.listdir(local_dir)):
        src = os.path.join(local_dir, name)
        if os.path.isfile(src):
            fs.copyFromLocalFile(False, True, jvm.org.apache.hadoop.fs.Path("file://" + os.path.abspath(src)), jvm.org.apache.hadoop.fs.Path(f"{hdfs_dir.rstrip('/')}/{name}"))

@dataclass
class Config:
    hdfs_work_dir: str = "/user/data/kshape/model"
    local_out_dir: str = "/workspace/code/kshape/model"
    tf_connector_jar: str = "/workspace/code/kshape/lib/spark-tensorflow-connector_2.12-1.11.0.jar"
    tfrecord_jar: str = "/workspace/code/kshape/lib/spark-tfrecord_2.12-0.7.0.jar"
    time_col: str = "pickup_bin_30m"
    loc_col: str = "PULocationID"
    target_col: str = "pickup_demand"
    split_col: str = "dataset_split"
    time_key_col: str = "time_key"
    sequence_features: tuple[str, ...] = ("pickup_demand", "ewma_output", "rolling_mean_24h", "day_of_week")
    valid_splits: tuple[str, ...] = ("train", "validation", "test")
    seq_window: int = 48
    batch_size: int = 128
    shuffle_buffer: int = 8192
    epochs: int = 1
    early_stopping_patience: int = 5
    random_state: int = 42
    ridge_alpha: float = 1.0
    xgb_num_workers: int = 3
    xgb_n_estimators: int = 10
    xgb_max_depth: int = 5
    xgb_learning_rate: float = 0.03
    xgb_subsample: float = 0.85
    xgb_colsample_bytree: float = 0.85
    pred_rows_per_file: int = 200_000
    def local(self, *parts: str) -> str: return os.path.join(self.local_out_dir, *parts)
    def hdfs(self, *parts: str) -> str: return "/".join([self.hdfs_work_dir.rstrip("/"), *parts])
    @property
    def spark_jars(self) -> str: return ",".join((self.tf_connector_jar, self.tfrecord_jar))
    @property
    def spark_cp(self) -> str: return ":".join((self.tf_connector_jar, self.tfrecord_jar))

@dataclass
class SequenceScalerStats:
    seq_mean: np.ndarray
    seq_std: np.ndarray
    y_min: float
    y_denom: float
    @classmethod
    def from_dict(cls, data: Dict[str, object]) -> "SequenceScalerStats":
        return cls(
            np.asarray(data["seq_mean"], dtype=np.float32),
            np.asarray(data["seq_std"], dtype=np.float32),
            float(data["y_min"]),
            float(data["y_denom"]),
        )
    def to_dict(self) -> Dict[str, object]:
        return {"seq_mean": self.seq_mean.tolist(), "seq_std": self.seq_std.tolist(), "y_min": float(self.y_min), "y_denom": float(self.y_denom)}

def build_spark(c: Config) -> SparkSession:
    stage("SPARK RUNTIME"); print("[SPARK JARS]", c.spark_jars)
    return (SparkSession.builder.appName("Distributed-Forecast-Train")
            .config("spark.jars", c.spark_jars)
            .config("spark.driver.extraClassPath", c.spark_cp)
            .config("spark.executor.extraClassPath", c.spark_cp)
            .config("spark.master", "yarn")
            .config("spark.submit.deployMode", "client")
            .config("spark.driver.host", "192.168.1.111")
            .config("spark.driver.bindAddress", "0.0.0.0")
            .getOrCreate())

class XGBoostTrainer:
    def __init__(self, spark: SparkSession, c: Config): self.spark, self.c, self.model = spark, c, None
    def run(self):
        stage("1/3 - TRAIN XGBOOST")
        full = train = part = pred = None
        try:
            full = self.spark.read.parquet(self.c.hdfs("prepared", "tabular"))
            train = full.filter(F.col(self.c.split_col) == "train")
            self.model = SparkXGBRegressor(
                features_col="features_vector", label_col=self.c.target_col, prediction_col="xgb_pred",
                num_workers=self.c.xgb_num_workers, n_estimators=self.c.xgb_n_estimators, max_depth=self.c.xgb_max_depth,
                learning_rate=self.c.xgb_learning_rate, subsample=self.c.xgb_subsample, colsample_bytree=self.c.xgb_colsample_bytree,
                objective="reg:squarederror", random_state=self.c.random_state, tree_method="hist", eval_metric="mae", missing=0.0,
            ).fit(train)
            self.model.write().overwrite().save(self.c.hdfs("models", "spark_xgb_model"))
            for split in self.c.valid_splits:
                part = full.filter(F.col(self.c.split_col) == split)
                pred = self.model.transform(part).select(self.c.time_col, self.c.time_key_col, self.c.loc_col, self.c.split_col, self.c.target_col, "xgb_pred")
                pred.write.mode("overwrite").parquet(self.c.hdfs("predictions", "xgb", split))
                cleanup(self.spark, part, pred); part = pred = None
        finally:
            self.model = None
            cleanup(self.spark, full, train, part, pred)

class CNNLSTMTrainer:
    def __init__(self, spark: SparkSession, c: Config, scaler: SequenceScalerStats):
        self.spark, self.c, self.scaler, self.model = spark, c, scaler, None
        self.n_features = len(c.sequence_features)

    def run(self):
        stage("2/3 - TRAIN CNN-LSTM")
        train_ds = val_ds = None
        try:
            self.model = self._build_model()
            train_ds, val_ds = self._dataset("train", True), self._dataset("validation", True)
            self.model.fit(train_ds, validation_data=val_ds, epochs=self.c.epochs, callbacks=[tf.keras.callbacks.EarlyStopping(patience=self.c.early_stopping_patience, restore_best_weights=True)], verbose=1)
            self.model.save(self.c.local("cnn_lstm.keras"))
            for split in self.c.valid_splits: self._predict(split)
        finally:
            self.model = None
            cleanup(self.spark, train_ds, val_ds, clear_tf=True)

    def _build_model(self) -> tf.keras.Model:
        inp = tf.keras.layers.Input((self.c.seq_window, self.n_features))
        x = tf.keras.layers.Conv1D(64, 3, padding="causal", activation="relu")(inp)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.20)(x)
        x = tf.keras.layers.LSTM(64)(x)
        x = tf.keras.layers.Dense(32, activation="relu")(x)
        out = tf.keras.layers.Dense(1, activation="linear")(x)
        model = tf.keras.Model(inp, out)
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mae", metrics=["mae"])
        return model

    def _dataset(self, split: str, supervised: bool) -> tf.data.Dataset:
        mean, std = tf.constant(self.scaler.seq_mean.reshape(1, -1), tf.float32), tf.constant(self.scaler.seq_std.reshape(1, -1), tf.float32)
        y_min, y_den = tf.constant(self.scaler.y_min, tf.float32), tf.constant(self.scaler.y_denom, tf.float32)
        spec = {
            self.c.loc_col: tf.io.FixedLenFeature([], tf.int64),
            self.c.time_key_col: tf.io.FixedLenFeature([], tf.string),
            self.c.split_col: tf.io.FixedLenFeature([], tf.string),
            self.c.target_col: tf.io.FixedLenFeature([1], tf.float32),
            "sequence_flat": tf.io.FixedLenFeature([self.c.seq_window * self.n_features], tf.float32),
        }
        def parse(example: tf.Tensor):
            ex = tf.io.parse_single_example(example, spec)
            seq = (tf.reshape(ex["sequence_flat"], [self.c.seq_window, self.n_features]) - mean) / std
            y = (ex[self.c.target_col][0] - y_min) / y_den
            return (seq, y) if supervised else (seq, y, ex)
        # hdfs_uri = self.spark._jsc.hadoopConfiguration().get("fs.defaultFS")
        # pattern = f"{hdfs_uri}{self.c.hdfs('prepared', 'tfrecords', split)}/part-*"
        # files = tf.io.gfile.glob(pattern)
        # if not files: raise RuntimeError(f"No TFRecord files found for split '{split}' at path: {pattern}")
        # ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE).map(parse, num_parallel_calls=tf.data.AUTOTUNE)
        # --- BẮT ĐẦU: CƠ CHẾ CACHE HDFS XUỐNG LOCAL ---
        local_dir = self.c.local("tfrecords_cache", split)
        os.makedirs(local_dir, exist_ok=True)
        
        jvm = self.spark._jvm
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(self.spark._jsc.hadoopConfiguration())
        hdfs_dir_path = jvm.org.apache.hadoop.fs.Path(self.c.hdfs('prepared', 'tfrecords', split))
        
        local_files = []
        if fs.exists(hdfs_dir_path):
            for file_status in fs.listStatus(hdfs_dir_path):
                name = file_status.getPath().getName()
                if name.startswith("part-"):
                    local_file = os.path.abspath(os.path.join(local_dir, name))
                    local_files.append(local_file)
                    
                    # Chỉ tải xuống nếu file chưa có ở Local (tiết kiệm thời gian chạy lại)
                    if not os.path.exists(local_file):
                        print(f"Đang cache '{name}' của tập {split} từ HDFS xuống máy...")
                        # Xử lý đường dẫn thân thiện với Windows/Hadoop
                        local_uri = "file:///" + local_file.replace("\\", "/").lstrip("/")
                        fs.copyToLocalFile(False, file_status.getPath(), jvm.org.apache.hadoop.fs.Path(local_uri), True)
                        
        if not local_files: 
            raise RuntimeError(f"Không có file dữ liệu nào ở HDFS: {hdfs_dir_path.toString()}")
            
        # Cho TensorFlow đọc từ file Local thay vì HDFS
        ds = tf.data.TFRecordDataset(local_files, num_parallel_reads=tf.data.AUTOTUNE).map(parse, num_parallel_calls=tf.data.AUTOTUNE)
        # --- KẾT THÚC SỬA ---
        if split == "train": ds = ds.shuffle(self.c.shuffle_buffer)
        return ds.batch(self.c.batch_size).prefetch(tf.data.AUTOTUNE)

    def _predict(self, split: str):
        if self.model is None: raise RuntimeError("CNN-LSTM model chưa khởi tạo")
        local_dir, ds, shard = self.c.local("tmp", f"cnn_preds_{split}"), None, 0
        reset_dir(local_dir)
        buf = {self.c.loc_col: [], self.c.time_key_col: [], self.c.split_col: [], self.c.target_col: [], "cnn_lstm_pred": []}
        try:
            ds = self._dataset(split, False)
            for batch_x, _, meta in ds:
                pred = nonneg(tf.squeeze(self.model(batch_x, training=False), -1).numpy() * self.scaler.y_denom + self.scaler.y_min)
                buf[self.c.loc_col].extend(meta[self.c.loc_col].numpy().astype(np.int64).tolist())
                buf[self.c.time_key_col].extend(x.decode("utf-8") for x in meta[self.c.time_key_col].numpy())
                buf[self.c.split_col].extend([split] * len(pred))
                buf[self.c.target_col].extend(meta[self.c.target_col].numpy().reshape(-1).astype(np.float32).tolist())
                buf["cnn_lstm_pred"].extend(pred.astype(np.float32).tolist())
                if len(buf[self.c.loc_col]) >= self.c.pred_rows_per_file: shard = self._flush(local_dir, buf, shard)
            if buf[self.c.loc_col]: self._flush(local_dir, buf, shard)
            copy_dir_to_hdfs(self.spark, local_dir, self.c.hdfs("predictions", "cnn_lstm", split))
        finally:
            for k in buf: buf[k].clear()
            cleanup(self.spark, ds)

    @staticmethod
    def _flush(local_dir: str, buf: Dict[str, List[object]], shard: int) -> int:
        pdf = pd.DataFrame(buf)
        try: pdf.to_parquet(os.path.join(local_dir, f"part-{shard:05d}.parquet"), index=False)
        finally: del pdf; gc.collect()
        for k in buf: buf[k].clear()
        return shard + 1

class EnsembleTrainer:
    def __init__(self, spark: SparkSession, c: Config):
        self.spark, self.c, self.model = spark, c, None
        self.assembler = VectorAssembler(inputCols=["xgb_pred", "cnn_lstm_pred"], outputCol="meta_features")

    def run(self) -> Dict[str, Dict[str, float | None]]:
        stage("3/3 - TRAIN RIDGE ENSEMBLE")
        out, val_base, val_vec, base, vec, pred = {}, None, None, None, None, None
        try:
            val_base = self._base("validation")
            val_vec = self.assembler.transform(val_base)
            self.model = LinearRegression(featuresCol="meta_features", labelCol=self.c.target_col, predictionCol="ensemble_pred", regParam=self.c.ridge_alpha).fit(val_vec)
            self.model.write().overwrite().save(self.c.hdfs("models", "spark_ridge_meta_model"))
            for split in self.c.valid_splits:
                base = self._base(split)
                vec = self.assembler.transform(base)
                pred = self.model.transform(vec).withColumn("ensemble_pred", F.greatest("ensemble_pred", F.lit(0.0)))
                pred.write.mode("overwrite").parquet(self.c.hdfs("predictions", "ensemble", split))
                out[f"xgb_{split}"], out[f"cnn_{split}"], out[f"ens_{split}"] = metrics(base, "xgb_pred", self.c.target_col), metrics(base, "cnn_lstm_pred", self.c.target_col), metrics(pred, "ensemble_pred", self.c.target_col)
                cleanup(self.spark, base, vec, pred); base = vec = pred = None
            with open(self.c.local("metrics.json"), "w", encoding="utf-8") as f: json.dump(out, f, indent=2)
            print(json.dumps(out, indent=2))
            return out
        finally:
            self.model = None
            cleanup(self.spark, val_base, val_vec, base, vec, pred)

    def _base(self, split: str) -> DataFrame:
        xgb_df = self.spark.read.parquet(self.c.hdfs("predictions", "xgb", split)).alias("xgb")
        cnn_df = self.spark.read.parquet(self.c.hdfs("predictions", "cnn_lstm", split)).alias("cnn")
        return cnn_df.join(xgb_df, on=[self.c.loc_col, self.c.time_key_col, self.c.split_col], how="inner").select(
            F.col(f"xgb.{self.c.time_col}").alias(self.c.time_col),
            F.col(f"cnn.{self.c.time_key_col}").alias(self.c.time_key_col),
            F.col(f"cnn.{self.c.loc_col}").alias(self.c.loc_col),
            F.col(f"cnn.{self.c.split_col}").alias(self.c.split_col),
            F.col(f"cnn.{self.c.target_col}").alias(self.c.target_col),
            F.col("xgb.xgb_pred").cast("double").alias("xgb_pred"),
            F.col("cnn.cnn_lstm_pred").cast("double").alias("cnn_lstm_pred"),
        )

class PipelineRunner:
    def __init__(self, c: Config):
        self.c = c
        mkdirs(c.local_out_dir, c.local("tmp"))
        seed_all(c.random_state)
        self.spark = build_spark(c)
        self.spark.sparkContext.setLogLevel("WARN")

    def _load_scaler(self) -> SequenceScalerStats:
        with open(self.c.local("sequence_scaler_stats.json"), "r", encoding="utf-8") as f:
            return SequenceScalerStats.from_dict(json.load(f))

    def run(self):
        scaler = self._load_scaler()
        XGBoostTrainer(self.spark, self.c).run()
        CNNLSTMTrainer(self.spark, self.c, scaler).run()
        metrics_out = EnsembleTrainer(self.spark, self.c).run()
        joblib.dump({"config": asdict(self.c), "sequence_scaler_stats": scaler.to_dict(), "metrics_path": self.c.local("metrics.json"), "metrics_keys": sorted(metrics_out)}, self.c.local("pipeline_metadata.joblib"))
        print("\n[PART 2 COMPLETE]")

if __name__ == "__main__":
    runner = None
    try:
        runner = PipelineRunner(Config())
        runner.run()
    finally:
        del runner
        gc.collect()


I0000 00:00:1775326676.287698   17693 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775326676.521654   17693 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775326679.403185   17693 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.10/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.10/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.10/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings


SPARK RUNTIME
[SPARK JARS] /workspace/code/kshape/lib/spark-tensorflow-connector_2.12-1.11.0.jar,/workspace/code/kshape/lib/spark-tfrecord_2.12-0.7.0.jar
:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-baf9d35c-43ff-46cb-8316-a21e508c001a;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 288ms :: artifacts dl 5ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------


1/3 - TRAIN XGBOOST


2026-04-04 18:18:20,759 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.85, 'device': 'cpu', 'eval_metric': 'mae', 'learning_rate': 0.03, 'max_depth': 5, 'random_state': 42, 'subsample': 0.85, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 10}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}
[18:18:49] [0]	training-mae:11.51221                                (0 + 3) / 3]
[18:18:49] [1]	training-mae:11.17439
[18:18:49] [2]	training-mae:10.84670
[18:18:50] [3]	training-mae:10.52924
[18:18:50] [4]	training-mae:10.22154
[18:18:50] [5]	training-mae:9.92354
[18:18:51] [6]	training-mae:9.63476
[18:18:51] [7]	training-mae:9.36773
[18:18:51] [8]	training-mae:9.09577
[18:18:52] [9]	training-mae:8.83237
2026-04-04 18:18:53,278 INFO XGBoost-PySpark: _fit Finished xgboost training!   


...

E0000 00:00:1775326754.811822   17693 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1775326757.145715   18362 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


  73728/Unknown 3050s 41ms/step - loss: 0.0019 - mae: 0.0019

/usr/local/lib/python3.10/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


73729/73729 ━━━━━━━━━━━━━━━━━━━━ 3377s 46ms/step - loss: 9.2487e-04 - mae: 9.2487e-04 - val_loss: 2.2750e-04 - val_mae: 2.2750e-04


KeyboardInterrupt: 